# WASB-SBDT Fine-Tuning on Modified Tennis Dataset

This notebook fine-tunes the pre-trained WASB model on a modified tennis dataset stored in `/kaggle/input/`.

## Strategy
- **Symlink** the read-only `/kaggle/input/` dataset to the repo's expected path (zero disk cost).
- **Inject** a custom `train.yaml` via `%%writefile` to configure fine-tuning hyperparameters.
- **Lower learning rate** (`1e-5`) to prevent catastrophic forgetting of pre-trained weights.
- Save all checkpoints to `/kaggle/working/outputs/` for easy download.

**Before running**: in the Kaggle Data panel, add your modified tennis dataset and note its folder name.

## Cell 1 — Environment Setup & Repository Clone

In [ ]:
import os

# Move to the working directory (read-write, 20 GB limit)
%cd /kaggle/working/

# Clone the WASB-SBDT repository
!git clone https://github.com/deepdewdeep/WASB-SBDT.git

# Install required Python dependencies
!pip install hydra-core omegaconf tqdm

## Cell 2 — Download Pre-Trained Weights & Symlink Dataset

> **Action required**: Replace `your-modified-tennis-dataset-name` below with the exact
> folder name shown in the Kaggle **Data** panel (e.g., `modified-tennis-v2`).

In [ ]:
%%bash
set -e
cd /kaggle/working/WASB-SBDT

# ── 1. Download pre-trained WASB weights ──────────────────────────────────────
cd src
bash setup_scripts/setup_weights.sh
cd ..

# ── 2. Create datasets directory ──────────────────────────────────────────────
mkdir -p datasets

# ── 3. Symlink Kaggle input dataset → repo expected path ─────────────────────
# Costs 0 bytes and avoids the 20 GB /kaggle/working/ disk limit.
# REPLACE the value below with your actual Kaggle dataset folder name.
KAGGLE_DATASET_NAME="your-modified-tennis-dataset-name"

ln -sfn "/kaggle/input/${KAGGLE_DATASET_NAME}" datasets/tennis

echo "Symlink created:"
ls -la datasets/tennis | head -n 8

## Cell 3 — Inject Fine-Tuning Configuration

We write a custom `train.yaml` that:
- Points Hydra output to `/kaggle/working/outputs/`.
- Sets a low learning rate (`1e-5`) to fine-tune without destroying pre-trained features.
- Runs for 30 epochs (adjust as needed).

In [ ]:
%%writefile /kaggle/working/WASB-SBDT/src/configs/train.yaml
defaults:
  - _self_
  - runner: train_and_test
  - dataset: tennis
  - model: wasb
  - dataloader: default
  - detector: tracknetv2
  - transform: default
  - tracker: online
  - optimizer: adam_multistep
  - loss: hm_wbce

# Enable training data loading
dataloader:
  train: true

# Fine-tuning: lower learning rate to preserve pre-trained features
optimizer:
  learning_rate: 0.00001

# Override runner settings for fine-tuning
runner:
  max_epochs: 30
  best_model_name: wasb_tennis_finetuned.pth.tar

# Override dataset root to match the symlinked Kaggle path
dataset:
  root_dir: /kaggle/working/WASB-SBDT/datasets/tennis

hydra:
  run:
    dir: /kaggle/working/outputs/${now:%Y-%m-%d_%H-%M-%S}

output_dir: /kaggle/working/outputs/finetuned
seed: 1234

## Cell 4 — Run the Fine-Tuning Pipeline

The pre-trained WASB tennis weights are loaded via `detector.model_path`.  
Checkpoints are saved every epoch to `/kaggle/working/outputs/finetuned/`.

In [ ]:
%%bash
set -e
cd /kaggle/working/WASB-SBDT/src

python3 main.py \
    --config-name=train \
    dataset=tennis \
    model=wasb \
    detector.model_path=/kaggle/working/WASB-SBDT/pretrained_weights/wasb_tennis_best.pth.tar

echo "Fine-tuning complete."
echo "Checkpoints saved to /kaggle/working/outputs/finetuned/"

## Cell 5 — Verify Outputs

List the saved checkpoints so you can download them from the Kaggle output panel.

In [ ]:
import glob

checkpoints = sorted(glob.glob('/kaggle/working/outputs/**/*.pth.tar', recursive=True))
if checkpoints:
    print(f'Found {len(checkpoints)} checkpoint(s):')
    for ckpt in checkpoints:
        print(' ', ckpt)
else:
    print('No checkpoints found yet. Check the training logs above for errors.')